# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv('data/AviationData.csv', encoding='latin1', low_memory=False)
print(df.shape)
df.head()

(88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

In [4]:
# NaN counts and percentages per column
nan_summary = pd.DataFrame({'n_nan': df.isna().sum(),
                            'pct_nan': (df.isna().mean() * 100).round(1)})
nan_summary.sort_values('pct_nan', ascending=False)

,n_nan,pct_nan
Schedule,76307,85.8
Air.carrier,72241,81.3
FAR.Description,56866,64.0
Aircraft.Category,56602,63.7
Latitude,54507,61.3
Longitude,54516,61.3
Airport.Code,38757,43.6
Airport.Name,36185,40.7
Broad.phase.of.flight,27165,30.6
Publication.Date,13771,15.5


In [5]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

**Filtering plan**

1. **Date** – convert `Event.Date` to datetime and keep events from 1983 onwards (40-year max make/model lifetime).
2. **Make / Model** – drop the handful of rows with no Make or Model (we cannot attribute them to an aircraft type). Standardise the text (strip whitespace, title-case Make, upper-case Model) so that the same aircraft is not split across spellings.
3. **Aircraft.Category** – ~66% of rows are NaN. *Imputation:* if the same `(Make, Model)` pair appears elsewhere with a recorded category, assign the most common recorded category for that pair (e.g. an un-categorised Cessna 172 is clearly an Airplane). Rows that still cannot be categorised are dropped, as are non-airplane categories (helicopters, gliders, balloons, ...).
4. **Amateur.Built** – the client only wants professional builds, so keep `Amateur.Built == 'No'` (the ~100 NaNs are dropped as we cannot confirm they are professional builds).

In [6]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df['Year'] = df['Event.Date'].dt.year
df = df[df['Year'] >= 1983].copy()
print('After date filter:', df.shape)

df = df.dropna(subset=['Make', 'Model'])
df['Make'] = df['Make'].str.strip().str.title()
df['Model'] = df['Model'].str.strip().str.upper()
print('After dropping missing Make/Model:', df.shape)

After date filter: (85289, 32)
After dropping missing Make/Model: (85197, 32)


In [7]:
df['Aircraft.Category'].value_counts(dropna=False)

Aircraft.Category
NaN                  56496
Airplane             24421
Helicopter            3149
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [8]:
# impute missing category from other records of the same Make/Model
known = df.dropna(subset=['Aircraft.Category'])
cat_lookup = known.groupby(['Make', 'Model'])['Aircraft.Category'].agg(lambda s: s.mode().iloc[0])

missing = df['Aircraft.Category'].isna()
df.loc[missing, 'Aircraft.Category'] = [cat_lookup.get(k, np.nan)
                                        for k in zip(df.loc[missing, 'Make'], df.loc[missing, 'Model'])]
print('Imputed categories for', missing.sum() - df['Aircraft.Category'].isna().sum(), 'rows')
df['Aircraft.Category'].value_counts(dropna=False).head(8)

Imputed categories for 43701 rows


Aircraft.Category
Airplane             64023
NaN                  12795
Helicopter            6782
Glider                 808
Balloon                305
Gyrocraft              164
Weight-Shift           162
Powered Parachute       92
Name: count, dtype: int64

In [9]:
df['Amateur.Built'].value_counts(dropna=False)

Amateur.Built
No     76888
Yes     8211
NaN       98
Name: count, dtype: int64

In [10]:
df = df[(df['Aircraft.Category'] == 'Airplane') & (df['Amateur.Built'] == 'No')].copy()
print('After airplane / professional-build filter:', df.shape)

After airplane / professional-build filter: (60686, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Assumptions**

- The four injury columns (`Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Minor.Injuries`, `Total.Uninjured`) are disjoint and together account for everyone on board. A NaN in one of these columns most likely means *none reported*, so NaNs are filled with 0.
- `Total.Passengers` = sum of the four columns (this is really "total occupants", crew included).
- Rows where `Total.Passengers == 0` carry no information about injury likelihood (nobody was recorded on board), so they are dropped.
- **Derived metric:** `Fatal.Serious.Frac = (Fatal + Serious) / Total.Passengers` — the estimated probability that a person on board was killed or seriously injured.

In [11]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
print(df[injury_cols].isna().sum())
df[injury_cols] = df[injury_cols].fillna(0)

df['Total.Passengers'] = df[injury_cols].sum(axis=1)
print('Rows with zero recorded occupants:', (df['Total.Passengers'] == 0).sum())
df = df[df['Total.Passengers'] > 0].copy()

df['Fatal.Serious.Frac'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']
df[['Total.Passengers', 'Fatal.Serious.Frac']].describe()

Total.Fatal.Injuries      7826
Total.Serious.Injuries    8591
Total.Minor.Injuries      8120
Total.Uninjured           3579
dtype: int64
Rows with zero recorded occupants: 1067


,Total.Passengers,Fatal.Serious.Frac
count,59619.000000,59619.000000
mean,6.253694,0.269039
std,28.055347,0.427950
min,1.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,2.000000,0.666667
max,588.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

`Aircraft.damage` has categories *Substantial, Destroyed, Minor, Unknown* and NaN. *Unknown* is merged into NaN. We do not impute damage (there is no reliable proxy), so the derived column `Destroyed` is 1 if destroyed, 0 if Minor/Substantial, and NaN when the damage is unknown — these rows still count for the injury metric but are ignored when averaging the destruction rate.

In [12]:
print(df['Aircraft.damage'].value_counts(dropna=False))
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)
df['Destroyed'] = np.where(df['Aircraft.damage'].isna(), np.nan,
                           (df['Aircraft.damage'] == 'Destroyed').astype(float))
df['Destroyed'].value_counts(dropna=False)

Aircraft.damage
Substantial    44746
Destroyed      11761
NaN             1539
Minor           1499
Unknown           74
Name: count, dtype: int64


Destroyed
0.0    46245
1.0    11761
NaN     1613
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Make cleaning tasks**

1. Case/whitespace already standardised above (`CESSNA` vs `Cessna`).
2. Remove punctuation and parenthetical notes, e.g. `Ercoupe (Eng & Research Corp.)` → `Ercoupe`.
3. Strip corporate suffixes (`Inc`, `Corp`, `Co`, `Company`, `Aircraft`, `Industries`, `Gmbh`, ...), e.g. `Cirrus Design Corp.` and `Air Tractor Inc` collapse onto `Cirrus Design` / `Air Tractor`.
4. Manually map remaining aliases of the same manufacturer, e.g. `Dehavilland` → `De Havilland`, `Cirrus Design` → `Cirrus`, `Grumman American` → `Grumman`, `Raytheon`/`Hawker Beechcraft` → `Beech`, `Gates Learjet` → `Learjet`.
5. Keep only Makes with at least 50 records so that per-make averages are based on a reasonable sample.

In [13]:
import re

def clean_make(m):
    m = re.sub(r'\(.*?\)', '', m)        # parenthetical notes
    m = re.sub(r'[.,]', '', m)           # punctuation
    m = re.sub(r'\b(Inc|Corp|Corporation|Co|Company|Ltd|Llc|Gmbh|Ag|Ab|As|Sa|Ind|Industries|Industrie|'
               r'International|Aircraft|Acft|Aviation|Avn|Mfg|Aerospace)\b', '', m)  # corporate suffixes
    return re.sub(r'\s+', ' ', m).strip(' -')

make_aliases = {
    'Dehavilland': 'De Havilland', 'Cirrus Design': 'Cirrus',
    'Grumman American': 'Grumman', 'Grumman-Schweizer': 'Grumman', 'Grumman Eng Cor-Schweizer': 'Grumman',
    'Raytheon': 'Beech', 'Hawker Beechcraft': 'Beech', 'Beechcraft': 'Beech',
    'American Champion': 'Champion', 'Gates Learjet': 'Learjet',
    'Cub Crafters': 'Cubcrafters', 'Gulfstream American': 'Gulfstream',
}

print('Unique makes before:', df['Make'].nunique())
df['Make'] = df['Make'].map(clean_make).replace(make_aliases)
print('Unique makes after:', df['Make'].nunique())

Unique makes before: 1068


Unique makes after: 849


In [14]:
make_counts = df['Make'].value_counts()
keep_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(keep_makes)].copy()
print(f'{len(keep_makes)} makes kept, {len(df)} rows')
df['Make'].value_counts()

49 makes kept, 57032 rows


Make
Cessna               24915
Piper                13649
Beech                 4635
Boeing                1570
Grumman               1345
Mooney                1222
Bellanca               916
Air Tractor            851
Champion               564
Aeronca                531
Maule                  488
Cirrus                 455
De Havilland           394
Stinson                392
Luscombe               366
Taylorcraft            328
Aero Commander         312
Rockwell               303
Mcdonnell Douglas      286
North American         285
Ayres                  229
Aviat                  223
Ercoupe                206
Airbus                 206
Embraer                173
Diamond                162
Learjet                157
Schweizer              140
Bombardier             127
Lake                   123
Douglas                119
Socata                 102
Waco                    99
American                88
Ryan                    86
Gulfstream              86
Swearingen             

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [15]:
print('Model NaNs:', df['Model'].isna().sum())   # already dropped above
# clean model labels: remove spaces and hyphens so 'PA 28-140', 'PA-28-140' and 'PA28140' are the same model
df['Model'] = df['Model'].str.replace(r'[\s\-]+', '', regex=True)
df['Model'].value_counts().head(15)

Model NaNs: 0


Model
152        2225
172        1637
172N       1096
PA28140     929
172M        789
150         748
172P        670
PA18        665
PA18150     622
182         614
180         595
PA28180     594
PA28161     560
150M        551
PA28181     549
Name: count, dtype: int64

In [16]:
# Are model labels unique to a make?
makes_per_model = df.groupby('Model')['Make'].nunique().sort_values(ascending=False)
print('Models shared by more than one make:', (makes_per_model > 1).sum())
makes_per_model.head(10)

Models shared by more than one make: 87


Model
500        4
NAVIONA    3
S2R        3
60         3
S2B        3
NAVION     3
DC1030     3
AA5        3
G164B      3
PA18150    2
Name: Make, dtype: int64

In [17]:
df[df['Model'].isin(makes_per_model.index[:3])].groupby(['Model', 'Make']).size()

Model    Make          
500      Aero Commander     18
         Cessna             26
         Embraer             1
         Rockwell            2
NAVIONA  Navion              3
         North American      9
         Ryan               16
S2R      Aero Commander     69
         Ayres              91
         Rockwell          165
dtype: int64

Model labels are **not** unique across makes (e.g. a bare `"180"` or `"A-1"` can belong to several manufacturers). We therefore build `Make.Model`, a combined identifier for a specific plane type.

In [18]:
df['Make.Model'] = df['Make'] + ' ' + df['Model']
print('Unique plane types:', df['Make.Model'].nunique())
df['Make.Model'].value_counts().head(10)

Unique plane types: 1708


Make.Model
Cessna 152       2225
Cessna 172       1637
Cessna 172N      1096
Piper PA28140     929
Cessna 172M       789
Cessna 150        748
Cessna 172P       670
Piper PA18        664
Piper PA18150     620
Cessna 182        614
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

**Cleaning tasks**

- `Engine.Type`: merge `UNK`, `NONE`, `LR` and NaN into `Unknown`; very rare types (Electric, Hybrid Rocket, Geared Turbofan) are folded into `Other`.
- `Weather.Condition`: `UNK` / `Unk` / NaN → `Unknown`. Valid values are VMC (visual) and IMC (instrument).
- `Number.of.Engines`: an airplane cannot have 0 engines — those are recording errors and are set to NaN.
- `Purpose.of.flight`: merge spelling variants (`Air Race show` / `Air Race/show`, the `Public Aircraft - *` variants and codes `PUBS` / `ASHO`) and NaN → `Unknown`.
- `Broad.phase.of.flight`: NaN → `Unknown` (NaN is common for newer reports where the field was discontinued).

In [19]:
for c in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[c].value_counts(dropna=False), '\n')

Engine.Type
Reciprocating      49184
NaN                 3467
Turbo Prop          2281
Turbo Fan           1400
Unknown              415
Turbo Jet            270
Turbo Shaft           13
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

Weather.Condition
VMC    49660
IMC     4427
NaN     2249
UNK      517
Unk      179
Name: count, dtype: int64 

Number.of.Engines
1.0    46498
2.0     7519
NaN     2598
3.0      188
4.0      147
0.0       82
Name: count, dtype: int64 

Purpose.of.flight
Personal                     33210
Instructional                 7851
NaN                           3413
Unknown                       3319
Aerial Application            3161
Business                      2677
Positioning                    925
Other Work Use                 500
Ferry                          479
Aerial Observation             371
Executive/corporate            300
Public Aircraft                255
Skydiving                      172
Flight Test                

In [20]:
df['Engine.Type'] = df['Engine.Type'].replace({'UNK': 'Unknown', 'NONE': 'Unknown', 'LR': 'Unknown'}).fillna('Unknown')
df.loc[df['Engine.Type'].isin(['Electric', 'Hybrid Rocket', 'Geared Turbofan']), 'Engine.Type'] = 'Other'

df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace({'UNK': 'Unknown'}).fillna('Unknown')

df.loc[df['Number.of.Engines'] == 0, 'Number.of.Engines'] = np.nan

df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race/show': 'Air Race show',
    'Public Aircraft - Federal': 'Public Aircraft', 'Public Aircraft - State': 'Public Aircraft',
    'Public Aircraft - Local': 'Public Aircraft',
    'PUBS': 'Public Aircraft', 'ASHO': 'Air Race show'}).fillna('Unknown')

df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')

for c in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines']:
    print(df[c].value_counts(dropna=False), '\n')

Engine.Type
Reciprocating    49184
Unknown           3883
Turbo Prop        2281
Turbo Fan         1400
Turbo Jet          270
Turbo Shaft         13
Other                1
Name: count, dtype: int64 

Weather.Condition
VMC        49660
IMC         4427
Unknown     2945
Name: count, dtype: int64 

Number.of.Engines
1.0    46498
2.0     7519
NaN     2680
3.0      188
4.0      147
Name: count, dtype: int64 



### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [21]:
nan_pct = df.isna().mean().sort_values(ascending=False) * 100
nan_pct.round(1)

Schedule                  87.7
Air.carrier               81.7
FAR.Description           67.8
Longitude                 60.7
Latitude                  60.7
Airport.Code              39.3
Airport.Name              36.6
Publication.Date          16.5
Report.Status              6.3
Number.of.Engines          4.7
Aircraft.damage            2.6
Destroyed                  2.6
Registration.Number        1.0
Country                    0.1
Location                   0.0
Total.Uninjured            0.0
Weather.Condition          0.0
Event.Id                   0.0
Year                       0.0
Broad.phase.of.flight      0.0
Total.Serious.Injuries     0.0
Total.Passengers           0.0
Fatal.Serious.Frac         0.0
Total.Minor.Injuries       0.0
Engine.Type                0.0
Total.Fatal.Injuries       0.0
Purpose.of.flight          0.0
Investigation.Type         0.0
Amateur.Built              0.0
Model                      0.0
Make                       0.0
Aircraft.Category          0.0
Injury.S

In [22]:
# drop columns with more than 50% missing values, plus identifiers / admin fields not used in the analysis
high_nan = nan_pct[nan_pct > 50].index.tolist()
admin_cols = ['Accident.Number', 'Registration.Number', 'Report.Status', 'Publication.Date', 'Amateur.Built']
df = df.drop(columns=high_nan + admin_cols)
print('Dropped:', high_nan + admin_cols)
df.info()

Dropped: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude', 'Accident.Number', 'Registration.Number', 'Report.Status', 'Publication.Date', 'Amateur.Built']
<class 'pandas.core.frame.DataFrame'>
Index: 57032 entries, 3601 to 88888
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                57032 non-null  object        
 1   Investigation.Type      57032 non-null  object        
 2   Event.Date              57032 non-null  datetime64[ns]
 3   Location                57008 non-null  object        
 4   Country                 56953 non-null  object        
 5   Airport.Code            34617 non-null  object        
 6   Airport.Name            36173 non-null  object        
 7   Injury.Severity         57032 non-null  object        
 8   Aircraft.damage         55539 non-null  object        
 9   Aircraft.Category       57032 non-null  object     

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [23]:
df.to_csv('data/AviationData_cleaned.csv', index=False)
print('Saved', df.shape)

Saved (57032, 26)
